In [1]:
from pathlib import Path
from datetime import datetime
import xarray as xr
import pandas as pd
import soundscapy as sspy

from scm_inst import scm

today = datetime.now().strftime("%Y-%m-%d")

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR.joinpath("data")
OUTPUT_DIR = PROJECT_DIR.joinpath("output/" + today)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_COLS = ["Language", "Institution", "Participant", "Recording"]

satp = sspy.databases.satp.load_zenodo()
satp.drop(columns=["Gender", "Age", "sequence_id", "loud"], inplace=True)

In [2]:
multiindex = pd.MultiIndex.from_frame(satp[INDEX_COLS])

satp.loc[multiindex.duplicated(), "Participant"] = "FER_5M04a"
multiindex = pd.MultiIndex.from_frame(satp[INDEX_COLS])
satp_multi = (
    satp.set_index(multiindex).drop(columns=INDEX_COLS).dropna(subset=scm.scale_abbrevs)
)


In [3]:
satp_xr = satp_multi.to_xarray()
satp_xr

<xarray.Dataset> Size: 382MB
Dimensions:      (Language: 18, Institution: 19, Participant: 646, Recording: 27)
Coordinates:
  * Language     (Language) object 144B 'arb' 'cmn' 'deu' ... 'tur' 'vie' 'zsm'
  * Institution  (Institution) object 152B 'BIS' 'CKU' 'FER' ... 'UPM' 'USP'
  * Participant  (Participant) object 5kB 'BIS_1' 'BIS_10' ... 'zsm_8' 'zsm_9'
  * Recording    (Recording) object 216B 'CG01' 'CG04' 'CG07' ... 'W22' 'W23a'
Data variables:
    PAQ1         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ2         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ3         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ4         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ5         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ6         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ7         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ8         (Language, Institution, Participant, Recording) float64 48MB ...

## Ipsatization

In [4]:
satp_xr = satp_xr - satp_xr.groupby("Participant").mean(
    skipna=True, dim=["Language", "Institution", "Recording"]
)
satp_xr

<xarray.Dataset> Size: 382MB
Dimensions:      (Language: 18, Institution: 19, Participant: 646, Recording: 27)
Coordinates:
  * Language     (Language) object 144B 'arb' 'cmn' 'deu' ... 'tur' 'vie' 'zsm'
  * Institution  (Institution) object 152B 'BIS' 'CKU' 'FER' ... 'UPM' 'USP'
  * Participant  (Participant) object 5kB 'BIS_1' 'BIS_10' ... 'zsm_8' 'zsm_9'
  * Recording    (Recording) object 216B 'CG01' 'CG04' 'CG07' ... 'W22' 'W23a'
Data variables:
    PAQ1         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ2         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ3         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ4         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ5         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ6         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ7         (Language, Institution, Participant, Recording) float64 48MB ...
    PAQ8         (Language, Institution, Participant, Recording) float64 48MB ...

## Step One: Tracey's Circular Order Model

In [5]:
def get_list_of_long_groupdfs(dataset: xr.Dataset, group: str) -> list[pd.DataFrame]:
    data_vars = list(dataset.data_vars)
    group_labels: list[str] = list(dataset[group].values)
    group_dfs: list[pd.DataFrame] = [
        dataset.sel({group: label}).to_dataframe().dropna().reset_index()[data_vars]
        for label in group_labels
    ]
    return group_dfs


def get_group_labels(dataset: xr.Dataset, group: str) -> list[str]:
    return list(dataset[group].values)


def scm_rthor_test(
    dataset: xr.Dataset, group: str, *, print_results=True
) -> pd.DataFrame:
    import rthor

    group_dfs = get_list_of_long_groupdfs(dataset, group)
    group_labels = get_group_labels(dataset, group)
    return rthor.test(
        group_dfs, order="circular8", labels=group_labels, print_results=print_results
    )

In [6]:
institution_rthor = scm_rthor_test(satp_xr, "Institution")

                                   RTHOR Test Results                                   
           19 matrices • 8 variables • 288 predictions • 40,320 permutations            
╭──────────┬────┬───────┬────────────────┬──────────────┬───────────────┬──────────────╮
│ Matrix   │    │    CI │ Interpretation │ Significance │     Satisfied │     Violated │
├──────────┼────┼───────┼────────────────┼──────────────┼───────────────┼──────────────┤
│ [1] BIS  │ ✓  │ 0.861 │ Excellent fit  │  p<.001 ***  │ 268/288 (93%) │  20/288 (7%) │
│ [2] CKU  │ ✓  │ 0.799 │ Excellent fit  │  p<.001 ***  │ 259/288 (90%) │ 29/288 (10%) │
│ [3] FER  │ ✓  │ 0.854 │ Excellent fit  │  p<.001 ***  │ 267/288 (93%) │  21/288 (7%) │
│ [4] FUU  │ ✓  │ 0.833 │ Excellent fit  │  p<.001 ***  │ 264/288 (92%) │  24/288 (8%) │
│ [5] HAN  │ ✓  │ 0.826 │ Excellent fit  │  p<.001 ***  │ 263/288 (91%) │  25/288 (9%) │
│ [6] ITB  │ ✓  │ 0.771 │ Excellent fit  │  p<.001 ***  │ 255/288 (89%) │ 33/288 (11%) │
│ [7] NTU  │ ↗  │ 0.674 │ Good fit       │   p<.01 **   │ 241/288 (84%) │ 47/288 (16%) │
│ [8] POT  │ ✓  │ 0.917 │ Excellent fit  │  p<.001 ***  │ 276/288 (96%) │  12/288 (4%) │
│ [9] RUG  │ ✓  │ 0.819 │ Excellent fit  │  p<.001 ***  │ 262/288 (91%) │  26/288 (9%) │
│ [10] SHU │ ↗  │ 0.681 │ Good fit       │   p<.01 **   │ 242/288 (84%) │ 46/288 (16%) │
│ [11] SJZ │ ✓  │ 0.806 │ Excellent fit  │  p<.001 ***  │ 260/288 (90%) │ 28/288 (10%) │
│ [12] STU │ ✓  │ 0.972 │ Excellent fit  │  p<.001 ***  │ 284/288 (99%) │   4/288 (1%) │
│ [13] TUB │ ✓  │ 0.889 │ Excellent fit  │  p<.001 ***  │ 272/288 (94%) │  16/288 (6%) │
│ [14] TUC │ ✓  │ 0.910 │ Excellent fit  │  p<.001 ***  │ 275/288 (95%) │  13/288 (5%) │
│ [15] UCL │ ✓  │ 0.986 │ Excellent fit  │  p<.001 ***  │ 286/288 (99%) │   2/288 (1%) │
│ [16] UGE │ ✓  │ 0.931 │ Excellent fit  │  p<.001 ***  │ 278/288 (97%) │  10/288 (3%) │
│ [17] UGR │ ✓  │ 0.910 │ Excellent fit  │  p<.001 ***  │ 275/288 (95%) │  13/288 (5%) │
│ [18] UPM │ ↗  │ 0.618 │ Good fit       │   p<.01 **   │ 233/288 (81%) │ 55/288 (19%) │
│ [19] USP │ ✓  │ 0.792 │ Excellent fit  │  p<.001 ***  │ 258/288 (90%) │ 30/288 (10%) │
╰──────────┴────┴───────┴────────────────┴──────────────┴───────────────┴──────────────╯
               ℹ️  Higher CI values indicate better fit (range: -1 to +1)                

In [7]:
language_rthor = scm_rthor_test(satp_xr, "Language")

                                   RTHOR Test Results                                   
           18 matrices • 8 variables • 288 predictions • 40,320 permutations            
╭──────────┬────┬───────┬────────────────┬──────────────┬───────────────┬──────────────╮
│ Matrix   │    │    CI │ Interpretation │ Significance │     Satisfied │     Violated │
├──────────┼────┼───────┼────────────────┼──────────────┼───────────────┼──────────────┤
│ [1] arb  │ ✓  │ 0.861 │ Excellent fit  │  p<.001 ***  │ 268/288 (93%) │  20/288 (7%) │
│ [2] cmn  │ ✓  │ 0.806 │ Excellent fit  │  p<.001 ***  │ 260/288 (90%) │ 28/288 (10%) │
│ [3] deu  │ ✓  │ 0.889 │ Excellent fit  │  p<.001 ***  │ 272/288 (94%) │  16/288 (6%) │
│ [4] ell  │ ✓  │ 0.910 │ Excellent fit  │  p<.001 ***  │ 275/288 (95%) │  13/288 (5%) │
│ [5] eng  │ ✓  │ 0.986 │ Excellent fit  │  p<.001 ***  │ 286/288 (99%) │   2/288 (1%) │
│ [6] fra  │ ✓  │ 0.931 │ Excellent fit  │  p<.001 ***  │ 278/288 (97%) │  10/288 (3%) │
│ [7] hrv  │ ✓  │ 0.854 │ Excellent fit  │  p<.001 ***  │ 267/288 (93%) │  21/288 (7%) │
│ [8] ind  │ ✓  │ 0.771 │ Excellent fit  │  p<.001 ***  │ 255/288 (89%) │ 33/288 (11%) │
│ [9] ita  │ ✓  │ 0.917 │ Excellent fit  │  p<.001 ***  │ 276/288 (96%) │  12/288 (4%) │
│ [10] jpn │ ✓  │ 0.833 │ Excellent fit  │  p<.001 ***  │ 264/288 (92%) │  24/288 (8%) │
│ [11] kor │ ✓  │ 0.826 │ Excellent fit  │  p<.001 ***  │ 263/288 (91%) │  25/288 (9%) │
│ [12] nld │ ✓  │ 0.819 │ Excellent fit  │  p<.001 ***  │ 262/288 (91%) │  26/288 (9%) │
│ [13] por │ ✓  │ 0.792 │ Excellent fit  │  p<.001 ***  │ 258/288 (90%) │ 30/288 (10%) │
│ [14] spa │ ✓  │ 0.910 │ Excellent fit  │  p<.001 ***  │ 275/288 (95%) │  13/288 (5%) │
│ [15] swe │ ✓  │ 0.972 │ Excellent fit  │  p<.001 ***  │ 284/288 (99%) │   4/288 (1%) │
│ [16] tur │ ✓  │ 0.799 │ Excellent fit  │  p<.001 ***  │ 259/288 (90%) │ 29/288 (10%) │
│ [17] vie │ ↗  │ 0.681 │ Good fit       │   p<.01 **   │ 242/288 (84%) │ 46/288 (16%) │
│ [18] zsm │ ↗  │ 0.653 │ Good fit       │   p<.01 **   │ 238/288 (83%) │ 50/288 (17%) │
╰──────────┴────┴───────┴────────────────┴──────────────┴───────────────┴──────────────╯
               ℹ️  Higher CI values indicate better fit (range: -1 to +1)                

## Step Two: Structural Equation Modelling (CircE)

In [8]:
from soundscapy.satp import SATP


ImportError: R package 'sn' is not installed. Please install it by running in R: install.packages('sn')

ImportError: Error accessing R installation: 
    Conversion rules for `rpy2.robjects` appear to be missing. Those
    rules are in a Python `contextvars.ContextVar`. This could be caused
    by multithreading code not passing context to the thread.
    Check rpy2's documentation about conversions.
    . Please ensure R is installed and correctly configured.